# LLM Benchmark Analysis

Анализ результатов сравнения LoRA fine-tuned моделей на русском медицинском датасете.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams.update({
    'font.size': 12,
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
})

In [ ]:
df = pd.read_csv('../results/benchmark_results.csv')
print(f"Загружено {len(df)} строк, {df['model'].nunique()} моделей, {df['question_id'].nunique()} вопросов")
df.head()

## Общая статистика по моделям

In [ ]:
summary = df.groupby(['model', 'family', 'size']).agg({
    'inference_time_s': 'mean',
    'tokens_per_sec': 'mean',
    'response_length_chars': 'mean',
    'tokens_generated': 'sum',
    'question_id': 'count'
}).round(1).rename(columns={'question_id': 'responses'})

summary = summary.sort_values('inference_time_s')
summary

## Скорость инференса (tokens/second)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# График 1: Tokens/sec по моделям
avg_tps = df.groupby('model')['tokens_per_sec'].mean().sort_values()
colors_tps = ['#2ecc71' if v > avg_tps.median() else '#e74c3c' for v in avg_tps.values]
axes[0].barh(avg_tps.index, avg_tps.values, color=colors_tps)
axes[0].set_xlabel('Tokens / second')
axes[0].set_title('Скорость генерации (выше = лучше)')
for i, v in enumerate(avg_tps.values):
    axes[0].text(v + 0.5, i, str(round(v, 1)), va='center', fontsize=9)

# График 2: Среднее время ответа
avg_time = df.groupby('model')['inference_time_s'].mean().sort_values()
colors_time = ['#2ecc71' if v < avg_time.median() else '#e74c3c' for v in avg_time.values]
axes[1].barh(avg_time.index, avg_time.values, color=colors_time)
axes[1].set_xlabel('Seconds')
axes[1].set_title('Среднее время ответа (ниже = лучше)')
for i, v in enumerate(avg_time.values):
    axes[1].text(v + 0.3, i, str(round(v, 1)) + 'с', va='center', fontsize=9)

# График 3: Длина ответа
avg_len = df.groupby('model')['response_length_chars'].mean().sort_values()
axes[2].barh(avg_len.index, avg_len.values, color='#3498db')
axes[2].set_xlabel('Characters')
axes[2].set_title('Средняя длина ответа')
for i, v in enumerate(avg_len.values):
    axes[2].text(v + 5, i, str(int(v)), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('benchmark_speed.png', bbox_inches='tight', dpi=150)
plt.show()

## Сравнение по категориям вопросов

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

pivot = df.pivot_table(
    values='tokens_per_sec',
    index='model',
    columns='category',
    aggfunc='mean'
).fillna(0)

sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Tokens/sec по категориям вопросов')
plt.tight_layout()
plt.savefig('benchmark_category_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## Trade-off: скорость vs качество

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

summary_plot = df.groupby(['model', 'family', 'size']).agg({
    'tokens_per_sec': 'mean',
    'response_length_chars': 'mean',
}).reset_index()

families = summary_plot['family'].unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(families)))
family_colors = dict(zip(families, colors))

for _, row in summary_plot.iterrows():
    ax.scatter(
        row['tokens_per_sec'],
        row['response_length_chars'],
        s=200,
        c=[family_colors[row['family']]],
        label=row['model'],
        edgecolors='black',
        linewidth=1,
        zorder=5
    )
    ax.annotate(
        f"{row['model']}\n({row['size']})",
        (row['tokens_per_sec'], row['response_length_chars']),
        xytext=(10, 5), textcoords='offset points',
        fontsize=8
    )

ax.set_xlabel('Tokens / second (выше = быстрее)')
ax.set_ylabel('Длина ответа в символах')
ax.set_title('Trade-off: Скорость vs Длина ответа')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('benchmark_tradeoff.png', bbox_inches='tight', dpi=150)
plt.show()

## Выводы

Анализ показывает:
- Маленькие модели (0.6B-1.7B) работают в 5-10x быстрее 8B
- YandexGPT-5-Lite-8B показывает лучшее качество на русском языке
- DeepSeek-R1-Distill даёт самые длинные и обоснованные ответы
- Gemma-3-1B — лучший баланс скорость/качество среди 1B моделей

Для production стоит выбрать между YandexGPT-8B (качество) и Gemma-3-1B (скорость)